# Live Inference

This notebook loads the trained mask-based steering model, builds the lane mask from a live frame, and predicts steering.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import torch
from PIL import Image
from jetcar.camera import open_camera, read_rgb_frame
from jetcar.training import SmallPilotNet, build_lane_mask
from torchvision import transforms


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallPilotNet(in_channels=1).to(device)
model_path = project_root / 'models' / 'pilotnet_mask_demo.pt'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
transform = transforms.Compose([
    transforms.Resize((96, 160)),
    transforms.ToTensor(),
])


In [ ]:
cap = open_camera(source='usb', device_index=0, width=1280, height=720, warmup_frames=12)
frame = read_rgb_frame(cap)
cap.release()

mask_image = build_lane_mask(Image.fromarray(frame))
x = transform(mask_image).unsqueeze(0).to(device)
with torch.no_grad():
    steering = float(model(x).item())

print('Predicted steering:', round(steering, 3))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(frame)
plt.title('Camera Frame')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(mask_image, cmap='gray')
plt.title(f'Mask Input, steering={steering:+.3f}')
plt.axis('off')
plt.tight_layout()
plt.show()
